# Pipeline Estimasi Produksi Padi dari Citra Drone RGB (versi revisi IJ-AI 31320)

Notebook ini menggantikan `Copy of padi.ipynb`. Setiap tahap menghasilkan **angka evaluasi nyata** yang bisa dilaporkan di naskah dan ditunjukkan ke reviewer.

| Bagian | Isi | Hasil untuk naskah |
|---|---|---|
| A | Persiapan dan struktur folder | — |
| B | Metadata citra: kamera, ketinggian, sudut kamera, GSD | Tabel parameter akuisisi (Metode) |
| C | Segmentasi petak sawah (YOLOv8-seg) | mAP box/mask, precision, recall |
| D | (Opsional) potongan petak dari foto baru untuk menambah data fase | Dataset fase tambahan |
| E | Klasifikasi fase tumbuh (YOLOv8-cls) | Akurasi, F1 per kelas, confusion matrix |
| F | Ekstraksi fitur per petak: luas, fase, indeks RGB | Tabel fitur |
| G | Regresi hasil panen dengan **berat panen nyata** (RF, SVR, KNN, LR, CNN) | RMSE, MAE, R², MAPE, learning curve CNN |

**Struktur folder di Google Drive** — notebook membaca langsung folder `Data Padi` sesuai strukturnya, tidak perlu dipindah:

```
MyDrive/Data Padi/
├── 1. umur 1 minggu/ … 7. pasca panen/   # potongan petak berlabel fase (label dari informasi petani) — Bagian E
├── drone data merauke/                    # foto asli drone (JPG; DNG diabaikan) — Bagian B, C, F
├── hasil drone di Merauke/                # foto asli drone; duplikat dengan folder lain disaring otomatis
├── seg_dataset/           # (dibuat nanti) ekspor Roboflow format "YOLOv8" berisi data.yaml — Bagian C
├── panen.csv              # (dibuat nanti) berat panen nyata per petak — Bagian G
└── hasil/                 # semua tabel & gambar keluaran (dibuat otomatis)
```

Prinsip kerja: **tidak ada data simulasi** di notebook ini. Bagian G hanya berjalan jika `panen.csv` berisi berat panen hasil penimbangan nyata.

## A. Persiapan

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q ultralytics

In [ ]:
import os, re, json, time, random, shutil, platform
from pathlib import Path
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
from PIL import Image

ROOT    = Path('/content/drive/MyDrive/Data Padi')
CITRA   = ROOT                                                    # path foto dicatat relatif terhadap folder ini
CITRA_DIRS = [ROOT / 'drone data merauke', ROOT / 'hasil drone di Merauke']   # folder berisi foto asli drone
HASIL   = ROOT / 'hasil'; HASIL.mkdir(parents=True, exist_ok=True)
RUNS    = '/content/runs'
SEED    = 42
random.seed(SEED); np.random.seed(SEED)

# Tujuh kelas fase, urutan DITETAPKAN di sini (bukan dari urutan abjad folder) -> mencegah bug label di notebook lama.
# Kunci = nama kelas, nilai = nama folder di Drive. Label ditentukan dari informasi petani tentang umur tanaman.
FASE_FOLDER = {
    'umur_1_minggu':      '1. umur 1 minggu',
    'umur_2_minggu':      '2. umur dua minggu',
    'umur_1_bulan':       '3. umur 1 bulan',
    'umur_2_bulan':       '4. umur 2 bulan',
    'umur_3_bulan_lebih': '5. umur 3 bulan lebih',
    'masa_panen':         '6. masa panen',
    'pasca_panen':        '7. pasca panen',
}
FASE = list(FASE_FOLDER)
FASE_ID = {f: i for i, f in enumerate(FASE)}
for f, d in FASE_FOLDER.items():
    assert (ROOT / d).is_dir(), f'Folder tidak ditemukan: {ROOT / d}'

import hashlib
def sidik(path, n=1 << 20):
    # Sidik jari file (ukuran + MD5 1 MB pertama) untuk mendeteksi foto ganda walaupun namanya berbeda/sama.
    path = Path(path)
    with open(path, 'rb') as fh: return f'{path.stat().st_size}-{hashlib.md5(fh.read(n)).hexdigest()}'

import torch, ultralytics
print('Python', platform.python_version(), '| torch', torch.__version__, '| ultralytics', ultralytics.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'tidak ada (CPU)')

## B. Metadata citra dan GSD

Luas petak dihitung dari **GSD** (ukuran satu piksel di tanah):

GSD = (lebar sensor × tinggi terbang) / (focal length × lebar citra dalam piksel)

Catatan penting:
- Tanggal di EXIF mengikuti jam kamera drone. Jika jam kamera tidak disetel, tanggal sebenarnya harus diambil dari log penerbangan atau catatan lapangan dan dicatat di Metode.
- Tinggi terbang diambil dari `RelativeAltitude` (tinggi dari titik take-off) di metadata XMP DJI, **bukan** `GPSAltitude` yang merupakan tinggi dari permukaan laut.
- Rumus GSD hanya berlaku untuk **foto tegak lurus (nadir)**, yaitu `GimbalPitchDegree` antara −80° dan −90° (toleransi 10°; galat luas < 2%). Foto miring (seperti Gambar 3 di naskah lama) tidak boleh dipakai untuk menghitung luas.

In [ ]:
# Lebar sensor (mm) per kode kamera DJI. FC300C = Phantom 3 Standard (sensor 1/2.3", data penelitian ini), FC330 = Phantom 4, FC6310 = Phantom 4 Pro/Adv.
SENSOR_MM = {'FC330': 6.17, 'FC220': 6.17, 'FC300C': 6.17, 'FC300S': 6.17, 'FC300X': 6.17,
             'FC6310': 13.2, 'FC6310S': 13.2, 'FC6360': 4.87}

def _xmp(text, tag):
    m = re.search(rf'drone-dji:{tag}="([+\-\d\.]+)"', text)
    return float(m.group(1)) if m else None

def dji_meta(path):
    path = Path(path)
    raw = path.read_bytes().decode('latin-1')
    img = Image.open(path); exif = img.getexif(); ifd = exif.get_ifd(0x8769)
    model = str(exif.get(0x0110, '')).strip('\x00 ').upper()
    focal = ifd.get(0x920A)
    focal = float(focal) if focal else None
    rel_alt = _xmp(raw, 'RelativeAltitude')
    pitch = _xmp(raw, 'GimbalPitchDegree')
    lat = _xmp(raw, 'GpsLatitude'); lon = _xmp(raw, 'GpsLongitude') or _xmp(raw, 'GpsLongtitude')
    taken = ifd.get(0x9003)
    sw = next((v for k, v in SENSOR_MM.items() if model == k or model.startswith(k)), None)
    gsd = (sw * rel_alt) / (focal * img.width) if (sw and rel_alt and focal) else None
    nadir = pitch is not None and abs(pitch + 90) <= 10   # toleransi 10°: galat luas < 2%
    return dict(file=path.name, kamera=model, waktu=taken, lebar_px=img.width, tinggi_px=img.height,
                focal_mm=focal, tinggi_terbang_m=rel_alt, gimbal_pitch=pitch, nadir=nadir,
                lat=lat, lon=lon, gsd_cm=None if gsd is None else gsd * 100)

semua = sorted([p for d in CITRA_DIRS for p in d.rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg'}])   # semua subfolder
files, dipakai, ganda = [], {}, []
for p in semua:                                   # buang duplikat (isi file identik) antarfolder
    k = sidik(p)
    if k in dipakai: ganda.append((str(p.relative_to(ROOT)), str(dipakai[k].relative_to(ROOT))))
    else: dipakai[k] = p; files.append(p)
pd.DataFrame(ganda, columns=['duplikat', 'sama_dengan']).to_csv(HASIL / 'B_foto_duplikat.csv', index=False)
print(f'{len(semua)} file JPG ditemukan, {len(ganda)} duplikat dibuang, {len(files)} foto unik dipakai')
meta = pd.DataFrame([dji_meta(p) for p in files])
meta.insert(0, 'subfolder', [str(p.parent.relative_to(CITRA)) for p in files])
meta.insert(1, 'path', [str(p.relative_to(CITRA)) for p in files])
meta.to_csv(HASIL / 'B_metadata_citra.csv', index=False)
print(f'{len(meta)} citra dari {meta.subfolder.nunique()} subfolder | nadir: {meta.nadir.sum()} | kamera: {meta.kamera.unique()}')
print(meta.groupby('subfolder').agg(jumlah=('file','size'), nadir=('nadir','sum'), tinggi_m=('tinggi_terbang_m','median'), gsd_cm=('gsd_cm','median')).round(2))
meta.head(10)

In [ ]:
# Ringkasan untuk tabel "parameter akuisisi" di Metode
nad = meta[meta.nadir]
print(nad[['tinggi_terbang_m', 'gsd_cm']].describe().round(2))
print('Rentang tanggal:', meta.waktu.min(), '->', meta.waktu.max())

## C. Segmentasi petak sawah (YOLOv8-seg)

1. Buat akun dan proyek baru di Roboflow (Instance Segmentation), unggah **40–60 foto nadir** (daftar foto nadir ada di `hasil/B_metadata_citra.csv`, kolom `nadir = True`), usahakan mewakili setiap tanggal penerbangan.
2. Gambar poligon setiap petak (fitur *Smart Polygon* mempercepat), satu kelas: `petak_sawah`.
3. Buat versi dataset dengan pembagian **70/15/15** (train/valid/test), lalu ekspor format **YOLOv8** ke `seg_dataset/`, atau unduh dengan sel di bawah.

API key dimasukkan lewat `getpass` agar **tidak tersimpan di notebook** (notebook lama menyimpan key secara terbuka).

In [ ]:
# Opsional: unduh dataset dari Roboflow
# !pip install -q roboflow
# from getpass import getpass
# from roboflow import Roboflow
# rf = Roboflow(api_key=getpass('Roboflow API key: '))
# ds = rf.workspace('NAMA_WORKSPACE').project('NAMA_PROYEK').version(1).download('yolov8', location=str(ROOT/'seg_dataset'))
SEG_YAML = str(ROOT / 'seg_dataset' / 'data.yaml')

In [ ]:
from ultralytics import YOLO
t0 = time.time()
seg = YOLO('yolov8s-seg.pt')
seg.train(data=SEG_YAML, epochs=100, imgsz=640, batch=8, seed=SEED, patience=30,
          project=RUNS, name='seg', exist_ok=True)
print(f'Waktu training segmentasi: {(time.time()-t0)/60:.1f} menit')
BEST_SEG = f'{RUNS}/seg/weights/best.pt'
shutil.copy(BEST_SEG, HASIL / 'model_segmentasi_best.pt')

In [ ]:
seg = YOLO(BEST_SEG)
m = seg.val(data=SEG_YAML, split='test', imgsz=640)
seg_metrik = pd.DataFrame([{
    'Precision (box)': m.box.mp, 'Recall (box)': m.box.mr, 'mAP50 (box)': m.box.map50, 'mAP50-95 (box)': m.box.map,
    'Precision (mask)': m.seg.mp, 'Recall (mask)': m.seg.mr, 'mAP50 (mask)': m.seg.map50, 'mAP50-95 (mask)': m.seg.map}]).round(4)
seg_metrik.to_csv(HASIL / 'C_metrik_segmentasi_test.csv', index=False)
seg_metrik.T

## D. (Opsional) Potongan petak dari foto untuk menambah data fase

Dataset fase sudah tersedia di tujuh folder `1. umur 1 minggu` … `7. pasca panen`. Sel ini **tidak wajib**. Jalankan hanya jika suatu kelas memiliki kurang dari ±30 potongan: sel ini memotong setiap petak hasil segmentasi ke `fase_belum_dilabeli/`, lalu potongan yang umurnya diketahui dari informasi petani dipindahkan ke folder fase yang sesuai.

Fungsi `petak_masks` di sel ini **tetap harus dijalankan** karena dipakai di Bagian F.

In [ ]:
BUAT_POTONGAN = False   # ubah ke True bila ingin membuat potongan baru untuk dilabeli
UNLAB = ROOT / 'fase_belum_dilabeli'

def petak_masks(model, img_bgr, conf=0.5):
    # Mengembalikan daftar (mask biner ukuran asli, bbox) dari poligon hasil prediksi.
    r = model.predict(img_bgr, imgsz=640, conf=conf, verbose=False)[0]
    out = []
    if r.masks is None: return out
    H, W = img_bgr.shape[:2]
    for poly, box in zip(r.masks.xy, r.boxes.xyxy.cpu().numpy()):
        if len(poly) < 3: continue
        mk = np.zeros((H, W), np.uint8)
        cv2.fillPoly(mk, [poly.astype(np.int32)], 1)   # poligon di koordinat asli -> luas akurat
        out.append((mk, box.astype(int)))
    return out

n = 0
for p in (files if BUAT_POTONGAN else []):
    UNLAB.mkdir(exist_ok=True)
    img = cv2.imread(str(p))
    for i, (mk, (x1, y1, x2, y2)) in enumerate(petak_masks(seg, img)):
        crop = img[y1:y2, x1:x2].copy(); crop[mk[y1:y2, x1:x2] == 0] = 0
        tag = str(p.relative_to(CITRA).with_suffix('')).replace('/', '__')
        if crop.size: cv2.imwrite(str(UNLAB / f'{tag}_petak{i:02d}.jpg'), crop); n += 1
print(f'{n} potongan petak disimpan di {UNLAB}' if BUAT_POTONGAN else 'Lewati pembuatan potongan (BUAT_POTONGAN = False)')

## E. Klasifikasi fase tumbuh (YOLOv8-cls)

Dataset diambil dari tujuh folder fase di `Data Padi`. Potongan yang isinya identik (duplikat, termasuk yang tercatat di dua kelas berbeda) dibuang dulu, lalu data dibagi **70/15/15 secara berstrata per kelas**.

Catatan untuk Metode: nama potongan lama (misalnya `36.JPG`) tidak mencatat foto asalnya, sehingga pembagian per foto asal tidak bisa dilakukan untuk data ini. Potongan dari foto yang sama bisa saja masuk ke data latih sekaligus data uji, sehingga akurasi bisa sedikit optimistis. Ini perlu disebut sebagai keterbatasan. Potongan baru dari Bagian D (bernama `<foto>_petakNN.jpg`) otomatis dibagi per foto asal.

In [ ]:
CLS_DS = Path('/content/cls_ds')
if CLS_DS.exists(): shutil.rmtree(CLS_DS)
rows, lihat, konflik = [], {}, []
for k, fase in enumerate(FASE):
    fl_all = sorted([p for p in (ROOT / FASE_FOLDER[fase]).rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}])
    fl = []
    for p in fl_all:
        h = sidik(p)
        if h in lihat:
            if lihat[h] != fase: konflik.append((str(p.relative_to(ROOT)), lihat[h], fase))
            continue
        lihat[h] = fase; fl.append(p)
    # foto asal: bagian sebelum "_petak" untuk potongan baru; potongan lama dianggap berdiri sendiri
    sumber = sorted({p.stem.split('_petak')[0] if '_petak' in p.stem else f'{fase}/{p.stem}' for p in fl})
    random.Random(SEED + k).shuffle(sumber)
    n_tr, n_va = int(round(.7 * len(sumber))), int(round(.15 * len(sumber)))
    grup = {s: 'train' for s in sumber[:n_tr]} | {s: 'val' for s in sumber[n_tr:n_tr + n_va]} | {s: 'test' for s in sumber[n_tr + n_va:]}
    for p in fl:
        split = grup[p.stem.split('_petak')[0] if '_petak' in p.stem else f'{fase}/{p.stem}']
        d = CLS_DS / split / f'{k}_{fase}'        # prefix angka = urutan FASE yang ditetapkan
        d.mkdir(parents=True, exist_ok=True); shutil.copy2(p, d / f'{len(rows):05d}_{p.name}')
        rows.append((fase, split))
for s in ['train', 'val', 'test']:
    for k, f in enumerate(FASE): (CLS_DS / s / f'{k}_{f}').mkdir(parents=True, exist_ok=True)
if konflik:
    print(f'PERHATIAN: {len(konflik)} potongan identik tercatat di dua kelas berbeda; dipertahankan di kelas pertama:')
    pd.DataFrame(konflik, columns=['file', 'kelas_dipakai', 'kelas_lain']).to_csv(HASIL / 'E_label_konflik.csv', index=False)
sebaran = pd.crosstab(pd.Series([r[0] for r in rows], name='fase'), pd.Series([r[1] for r in rows], name='split'))
sebaran = sebaran.reindex(FASE); sebaran['total'] = sebaran.sum(1)
sebaran.to_csv(HASIL / 'E_sebaran_dataset_fase.csv'); sebaran

In [ ]:
t0 = time.time()
cls = YOLO('yolov8s-cls.pt')
cls.train(data=str(CLS_DS), epochs=100, imgsz=224, batch=16, seed=SEED, patience=30,
          project=RUNS, name='cls', exist_ok=True)
print(f'Waktu training klasifikasi: {(time.time()-t0)/60:.1f} menit')
BEST_CLS = f'{RUNS}/cls/weights/best.pt'
shutil.copy(BEST_CLS, HASIL / 'model_klasifikasi_best.pt')

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score
cls = YOLO(BEST_CLS)
idx_of = {v: k for k, v in cls.names.items()}
y_true, y_pred = [], []
for d in sorted((CLS_DS / 'test').iterdir()):
    for p in sorted(d.iterdir()):
        y_true.append(idx_of[d.name]); y_pred.append(int(cls.predict(str(p), imgsz=224, verbose=False)[0].probs.top1))
label = [cls.names[i] for i in sorted(cls.names)]
print('Akurasi test:', round(accuracy_score(y_true, y_pred), 4))
rep = classification_report(y_true, y_pred, labels=sorted(cls.names), target_names=label, digits=4, output_dict=True, zero_division=0)
pd.DataFrame(rep).T.round(4).to_csv(HASIL / 'E_metrik_klasifikasi_test.csv')
print(classification_report(y_true, y_pred, labels=sorted(cls.names), target_names=label, digits=4, zero_division=0))
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred, labels=sorted(cls.names)), display_labels=label).plot(xticks_rotation=30, cmap='Blues')
plt.title('Confusion matrix klasifikasi fase (data uji)'); plt.tight_layout()
plt.savefig(HASIL / 'E_confusion_matrix.png', dpi=300); plt.show()

### E2. Cross-validation 5-fold klasifikasi fase (disarankan untuk dilaporkan di naskah)

Data uji tunggal hanya 23 potongan, sehingga satu prediksi salah menggeser akurasi sekitar 4%. Sel ini menilai model dengan **5-fold cross-validation berstrata**: setiap potongan menjadi data uji tepat satu kali. Di dalam setiap fold, 15% data latih disisihkan sebagai validasi untuk memilih epoch terbaik, sehingga fold uji tidak pernah dipakai untuk memilih model.

Hasil: akurasi dan macro-F1 (rata-rata ± simpangan baku antar-fold), precision/recall/F1 per kelas dari prediksi gabungan, dan confusion matrix gabungan. Waktu: sekitar 5 × waktu training Bagian E.

In [ ]:
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score
from ultralytics import YOLO

# Kumpulkan semua potongan unik (duplikat sudah disaring di sel pembagian data Bagian E)
items = []
for d in sorted(CLS_DS.glob('*/*')):
    if d.is_dir():
        k = int(d.name.split('_', 1)[0])
        items += [(p, k) for p in sorted(d.iterdir()) if p.is_file()]
paths = np.array([str(p) for p, _ in items]); ys = np.array([k for _, k in items])
print(len(paths), 'potongan, per kelas:', np.bincount(ys, minlength=len(FASE)))

K = 5
skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=SEED)
y_true_all, y_pred_all, per_fold = [], [], []
t_all = time.time()
for f, (tr, te) in enumerate(skf.split(paths, ys)):
    tr_i, va_i = train_test_split(tr, test_size=0.15, stratify=ys[tr], random_state=SEED)
    root = Path(f'/content/cv_cls/fold{f}')
    if root.exists(): shutil.rmtree(root)
    for split, idx in [('train', tr_i), ('val', va_i), ('test', te)]:
        for k, fase in enumerate(FASE): (root / split / f'{k}_{fase}').mkdir(parents=True, exist_ok=True)
        for i in idx:
            shutil.copy2(paths[i], root / split / f'{ys[i]}_{FASE[ys[i]]}' / Path(paths[i]).name)
    m = YOLO('yolov8s-cls.pt')
    m.train(data=str(root), epochs=100, imgsz=224, batch=16, seed=SEED, patience=30,
            project='/content/runs_cv', name=f'fold{f}', exist_ok=True, verbose=False, plots=False)
    best = YOLO(f'/content/runs_cv/fold{f}/weights/best.pt')
    idx_of = {v: kk for kk, v in best.names.items()}
    yt, yp = [], []
    for d in sorted((root / 'test').iterdir()):
        for p in sorted(d.iterdir()):
            yt.append(idx_of[d.name]); yp.append(int(best.predict(str(p), imgsz=224, verbose=False)[0].probs.top1))
    acc = accuracy_score(yt, yp); mf1 = f1_score(yt, yp, average='macro', labels=list(range(len(FASE))), zero_division=0)
    per_fold.append(dict(fold=f + 1, n_test=len(yt), akurasi=acc, macro_f1=mf1))
    y_true_all += yt; y_pred_all += yp
    print(f'Fold {f+1}: akurasi {acc:.3f}, macro-F1 {mf1:.3f} (n uji = {len(yt)})')

cv = pd.DataFrame(per_fold)
print(f"\nAkurasi CV: {cv.akurasi.mean():.3f} ± {cv.akurasi.std():.3f} | macro-F1 CV: {cv.macro_f1.mean():.3f} ± {cv.macro_f1.std():.3f}")
print(f'Waktu total CV: {(time.time() - t_all) / 60:.1f} menit')
cv.round(4).to_csv(HASIL / 'E2_cv_per_fold.csv', index=False)
label = [f.replace('_', ' ') for f in FASE]
rep = classification_report(y_true_all, y_pred_all, labels=list(range(len(FASE))), target_names=label, digits=4, output_dict=True, zero_division=0)
pd.DataFrame(rep).T.round(4).to_csv(HASIL / 'E2_cv_metrik_per_kelas.csv')
print(classification_report(y_true_all, y_pred_all, labels=list(range(len(FASE))), target_names=label, digits=4, zero_division=0))
ConfusionMatrixDisplay(confusion_matrix(y_true_all, y_pred_all, labels=list(range(len(FASE)))), display_labels=label).plot(xticks_rotation=35, cmap='Blues')
plt.title('Confusion matrix klasifikasi fase (5-fold CV, gabungan)'); plt.tight_layout()
plt.savefig(HASIL / 'E2_cv_confusion_matrix.png', dpi=300); plt.show()

## F. Ekstraksi fitur per petak

Hanya foto **nadir** yang dipakai. Indeks vegetasi dihitung dari kanal tampak (RGB), karena kamera drone tidak memiliki kanal NIR:

- ExG = 2g − r − b (koordinat kromatik r, g, b = R, G, B dibagi R+G+B)
- VARI = (G − R) / (G + R − B)
- GLI = (2G − R − B) / (2G + R + B)
- NGRDI = (G − R) / (G + R)

Setiap foto juga disimpan dalam versi bernomor petak (`hasil/F_overlay/`), untuk mencocokkan nomor petak di foto dengan ID petak di catatan panen.

In [ ]:
EPS = 1e-6
def indeks_rgb(img_bgr, mk):
    px = img_bgr[mk.astype(bool)].astype(np.float64)
    B, G, R = px[:, 0], px[:, 1], px[:, 2]
    s = R + G + B + EPS
    return dict(ExG=np.mean(2 * G / s - R / s - B / s),
                VARI=np.mean(np.clip((G - R) / (G + R - B + EPS), -1, 1)),
                GLI=np.mean((2 * G - R - B) / (2 * G + R + B + EPS)),
                NGRDI=np.mean((G - R) / (G + R + EPS)))

OVL = HASIL / 'F_overlay'; OVL.mkdir(exist_ok=True)
fitur = []
for _, mrow in meta[meta.nadir & meta.gsd_cm.notna()].iterrows():
    img = cv2.imread(str(CITRA / mrow.path)); ov = img.copy()
    gsd_m = mrow.gsd_cm / 100
    for i, (mk, (x1, y1, x2, y2)) in enumerate(petak_masks(seg, img)):
        crop = img[y1:y2, x1:x2].copy(); crop[mk[y1:y2, x1:x2] == 0] = 0
        pr = cls.predict(crop, imgsz=224, verbose=False)[0].probs
        fase = cls.names[int(pr.top1)].split('_', 1)[1]
        ys, xs = np.nonzero(mk)
        fitur.append(dict(file=mrow.path, petak_idx=i, luas_m2=mk.sum() * gsd_m ** 2, gsd_cm=mrow.gsd_cm,
                          fase=fase, fase_id=FASE_ID[fase], fase_conf=float(pr.top1conf),
                          cx_px=int(xs.mean()), cy_px=int(ys.mean()), **indeks_rgb(img, mk)))
        cv2.drawContours(ov, cv2.findContours(mk, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0], -1, (0, 255, 255), 3)
        cv2.putText(ov, str(i), (int(xs.mean()), int(ys.mean())), cv2.FONT_HERSHEY_SIMPLEX, 3, (0, 0, 255), 8)
    cv2.imwrite(str(OVL / mrow.path.replace('/', '__')), ov)
fitur = pd.DataFrame(fitur)
fitur.to_csv(HASIL / 'F_fitur_petak.csv', index=False)
print(f'{len(fitur)} petak dari {fitur.file.nunique()} foto | total luas {fitur.luas_m2.sum():,.1f} m²')
fitur.head()

## G. Regresi hasil panen dengan data panen nyata

Isi file **`panen.xlsx`** (template disediakan) dan unggah ke folder `Data Padi`:

- sheet **`petak_lapangan`**: satu baris per petak sawah di lapangan, berisi `petak_id`, `tanggal_panen`, `berat_kg`, `luas_tertimbang_m2`, `jenis_gabah` (GKP/GKG), dan `sumber_data`, semuanya dari **bukti panen asli**;
- sheet **`pencocokan`**: daftar semua petak yang terdeteksi di foto (dari Bagian F). Isi `petak_id` untuk petak yang dikenali di gambar `hasil/F_overlay/`, dan `utuh` = Ya/Tidak.

Hanya deteksi yang **utuh** (tidak terpotong tepi foto) yang dipakai. Fitur citra dirata-ratakan per petak lapangan dari semua foto yang memuatnya.

Target model adalah **produktivitas (ton/ha)**, sedangkan produksi petak = produktivitas × luas (MAPE produktivitas sama dengan MAPE produksi bila luas yang dipakai sama). Dengan cara ini, model tidak sekadar "belajar" dari luas petak. Karena jumlah petak kecil, evaluasi memakai **5-fold cross-validation diulang 10 kali**, dan hyperparameter dipilih di dalam setiap fold (*nested*), sehingga data uji tidak pernah dipakai untuk tuning.

In [ ]:
# panen.xlsx: sheet 'petak_lapangan' (1 baris/petak lapangan) + sheet 'pencocokan' (1 baris/deteksi petak di foto)
if 'fitur' not in globals(): fitur = pd.read_csv(HASIL / 'F_fitur_petak.csv')
lap = pd.read_excel(ROOT / 'panen.xlsx', sheet_name='petak_lapangan').dropna(subset=['petak_id', 'berat_kg'])
coc = pd.read_excel(ROOT / 'panen.xlsx', sheet_name='pencocokan').dropna(subset=['petak_id'])
coc['utuh_bool'] = coc['utuh'].astype(str).str.lower().str.startswith('y')
# Fitur spektral & fase boleh diambil dari deteksi terpotong tepi foto bila luas acuan lapangan tersedia;
# kolom 'utuh' dipakai untuk validasi luas di sel berikutnya.
data = coc[['file', 'petak_idx', 'petak_id', 'utuh_bool']].merge(fitur, on=['file', 'petak_idx'], how='inner')
print(f"{len(lap)} petak lapangan berberat | {len(coc)} deteksi dicocokkan -> {data.petak_id.nunique()} petak punya fitur citra")
FITUR = ['fase_id', 'ExG', 'VARI', 'GLI', 'NGRDI']
agg = {c: 'mean' for c in FITUR if c != 'fase_id'} | {'fase_id': lambda s: s.mode().iloc[0], 'luas_m2': 'median', 'file': 'count'}
citra = data.groupby('petak_id').agg(agg).rename(columns={'file': 'n_foto'}).reset_index()
df = lap.merge(citra, on='petak_id', how='inner')
# luas acuan: luas yang ditimbang (ubinan/petak) bila diisi, jika tidak memakai luas petak dari citra
df['luas_ref_m2'] = df['luas_tertimbang_m2'].fillna(df['luas_m2'])
df['hasil_t_ha'] = df['berat_kg'] / df['luas_ref_m2'] * 10
print(f'{len(df)} petak unik untuk regresi | jenis gabah: {df.jenis_gabah.unique()} | sumber: {df.sumber_data.unique()}')
if df.jenis_gabah.nunique() > 1: print('PERINGATAN: GKP dan GKG tercampur; samakan dulu (konversi kadar air) sebelum regresi.')
if len(df) < 15: print('PERINGATAN: kurang dari 15 petak; laporkan hasil sebagai studi awal dengan keterbatasan jumlah sampel.')
df.to_csv(HASIL / 'G_dataset_regresi.csv', index=False)
print('\nStatistik produktivitas (t/ha):'); print(df['hasil_t_ha'].describe().round(3))
if df['hasil_t_ha'].std() < 0.1 * df['hasil_t_ha'].mean():
    print('CATATAN: variasi produktivitas antarpetak sangat kecil (koefisien variasi < 10%); '
          'R2 regresi akan rendah karena variasi target kecil; bandingkan model dengan baseline rata-rata.')
df.describe().round(3)

### G1. Validasi luas petak: segmentasi vs luas menurut petani

Membandingkan luas hasil segmentasi dengan luas lapangan untuk petak yang ditandai **utuh = Ya** (tidak terpotong tepi foto). Jika belum ada yang ditandai utuh, semua deteksi ditampilkan, dengan catatan bahwa petak terpotong tepi akan tampak lebih kecil.

In [ ]:
v = data.merge(lap[['petak_id', 'luas_tertimbang_m2']], on='petak_id').dropna(subset=['luas_tertimbang_m2'])
sub = v[v.utuh_bool] if v.utuh_bool.any() else v
if not v.utuh_bool.any(): print('Belum ada deteksi bertanda utuh = Ya; menampilkan semua deteksi (petak terpotong tepi ikut dihitung).')
lv = sub.groupby('petak_id').agg(luas_citra_m2=('luas_m2', 'median'), luas_lapangan_m2=('luas_tertimbang_m2', 'first'), n_foto=('file', 'count')).reset_index()
lv['selisih_persen'] = 100 * (lv.luas_citra_m2 - lv.luas_lapangan_m2) / lv.luas_lapangan_m2
mape_luas = lv.selisih_persen.abs().mean()
print(f'MAPE luas (n = {len(lv)} petak): {mape_luas:.1f}% | median selisih absolut: {lv.selisih_persen.abs().median():.1f}%')
lv.round(1).to_csv(HASIL / 'G1_validasi_luas.csv', index=False)
lv.round(1)

### G2. Estimasi produksi berbasis luas dan validasi terhadap produksi menurut petani

Produksi petak diestimasi sebagai **luas hasil segmentasi × produktivitas acuan**. Agar tidak melingkar, produktivitas acuan untuk setiap petak dihitung dari **rata-rata produktivitas petak-petak lain** (*leave-one-out*), bukan dari petak itu sendiri. Estimasi lalu dibandingkan dengan berat GKP yang dilaporkan petani (MAPE, RMSE, MAE). Hasil dihitung untuk semua petak dan untuk petak utuh saja.

In [ ]:
pp = data.groupby('petak_id').agg(luas_citra_m2=('luas_m2', 'median'), utuh=('utuh_bool', 'max')).reset_index()
pp = pp.merge(lap[['petak_id', 'berat_kg', 'luas_tertimbang_m2']], on='petak_id')
pp['prod_t_ha'] = pp.berat_kg / pp.luas_tertimbang_m2 * 10
tot, n = pp.prod_t_ha.sum(), len(pp)
pp['prod_acuan_loo_t_ha'] = (tot - pp.prod_t_ha) / (n - 1)
pp['estimasi_kg'] = pp.luas_citra_m2 * pp.prod_acuan_loo_t_ha / 10
pp['galat_persen'] = 100 * (pp.estimasi_kg - pp.berat_kg) / pp.berat_kg
def ringkas(d, label):
    e = d.estimasi_kg - d.berat_kg
    return dict(subset=label, n=len(d), MAPE_persen=round(d.galat_persen.abs().mean(), 1),
                MAE_kg=round(e.abs().mean(), 1), RMSE_kg=round(float(np.sqrt((e ** 2).mean())), 1),
                total_estimasi_kg=round(d.estimasi_kg.sum()), total_petani_kg=round(d.berat_kg.sum()),
                selisih_total_persen=round(100 * (d.estimasi_kg.sum() - d.berat_kg.sum()) / d.berat_kg.sum(), 1))
rk = [ringkas(pp, 'semua petak')] + ([ringkas(pp[pp.utuh], 'petak utuh')] if pp.utuh.any() else [])
rk = pd.DataFrame(rk); rk.to_csv(HASIL / 'G2_validasi_produksi_ringkas.csv', index=False)
pp.round(2).to_csv(HASIL / 'G2_validasi_produksi_per_petak.csv', index=False)
print(rk.to_string(index=False))
lim = [0, max(pp.berat_kg.max(), pp.estimasi_kg.max()) * 1.1]
plt.figure(figsize=(4.5, 4.5))
plt.scatter(pp.berat_kg, pp.estimasi_kg, c=np.where(pp.utuh, 'tab:blue', 'tab:orange'))
plt.plot(lim, lim, 'k--', lw=1); plt.xlim(lim); plt.ylim(lim)
plt.xlabel('Farmer-reported production (kg GKP)'); plt.ylabel('Estimated production (kg)')
plt.title('Area-based production estimate'); plt.tight_layout()
plt.savefig(HASIL / 'G2_estimasi_vs_petani.png', dpi=300); plt.show()
pp.round(1)

In [ ]:
from sklearn.model_selection import RepeatedKFold, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error

X = df[FITUR].to_numpy(float); y = df['hasil_t_ha'].to_numpy(float); luas = df['luas_ref_m2'].to_numpy(float)
inner = lambda: KFold(3, shuffle=True, random_state=SEED)
MODEL = {
  'RF':  (Pipeline([('m', RandomForestRegressor(random_state=SEED))]),
          {'m__n_estimators': [100, 300], 'm__max_depth': [None, 3, 5], 'm__min_samples_leaf': [1, 2, 4]}),
  'SVR': (Pipeline([('s', StandardScaler()), ('m', SVR(kernel='rbf'))]),
          {'m__C': [1, 10, 100], 'm__gamma': ['scale', 0.1, 0.01], 'm__epsilon': [0.05, 0.2]}),
  'KNN': (Pipeline([('s', StandardScaler()), ('m', KNeighborsRegressor())]),
          {'m__n_neighbors': [2, 3, 5], 'm__weights': ['uniform', 'distance']}),
  'LR':  (Pipeline([('s', StandardScaler()), ('p', PolynomialFeatures(include_bias=False)), ('m', LinearRegression())]),
          {'p__degree': [1, 2]}),
}

In [ ]:
# CNN 1D untuk fitur tabular (sama dengan pendekatan naskah: input berbentuk (sampel, fitur, 1))
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

def build_cnn(n_feat, filters=32, kernel=2, dense=32, lr=1e-3):
    tf.keras.utils.set_random_seed(SEED)
    m = keras.Sequential([keras.Input((n_feat, 1)),
                          layers.Conv1D(filters, kernel, activation='relu', padding='same'),
                          layers.Flatten(), layers.Dense(dense, activation='relu'), layers.Dense(1)])
    m.compile(optimizer=keras.optimizers.Adam(lr), loss='mse')
    return m

CNN_GRID = [dict(filters=f, kernel=k, lr=l) for f in (16, 32) for k in (2, 3) for l in (1e-3, 1e-2)]
ES = lambda: keras.callbacks.EarlyStopping(patience=30, restore_best_weights=True)

def fit_cnn(Xtr, ytr):
    # Pilih konfigurasi CNN dengan validasi internal (20% data latih), lalu latih ulang pada seluruh data latih.
    sc = StandardScaler().fit(Xtr); Z = sc.transform(Xtr)[..., None]
    cut = max(1, int(.2 * len(Z))); idx = np.random.RandomState(SEED).permutation(len(Z))
    va, tr = idx[:cut], idx[cut:]
    best = None
    for g in CNN_GRID:
        m = build_cnn(Z.shape[1], **g)
        h = m.fit(Z[tr], ytr[tr], validation_data=(Z[va], ytr[va]), epochs=500, batch_size=8, verbose=0, callbacks=[ES()])
        v = min(h.history['val_loss'])
        if best is None or v < best[0]: best = (v, g, len(h.history['val_loss']))
    m = build_cnn(Z.shape[1], **best[1]); m.fit(Z, ytr, epochs=best[2], batch_size=8, verbose=0)
    return sc, m, best[1]

In [ ]:
R, K = 10, 5
cv = RepeatedKFold(n_splits=K, n_repeats=R, random_state=SEED)
oof = {n: np.zeros((R, len(y))) for n in list(MODEL) + ['CNN']}
waktu = {n: 0.0 for n in oof}; param_terpilih = {n: [] for n in oof}

for f, (tr, te) in enumerate(cv.split(X)):
    r = f // K
    for n, (pipe, grid) in MODEL.items():
        t0 = time.time()
        gs = GridSearchCV(pipe, grid, cv=inner(), scoring='neg_root_mean_squared_error').fit(X[tr], y[tr])
        oof[n][r, te] = gs.predict(X[te]); waktu[n] += time.time() - t0; param_terpilih[n].append(gs.best_params_)
    t0 = time.time()
    sc, m, g = fit_cnn(X[tr], y[tr])
    oof['CNN'][r, te] = m.predict(sc.transform(X[te])[..., None], verbose=0).ravel()
    waktu['CNN'] += time.time() - t0; param_terpilih['CNN'].append(g)
    print(f'fold {f+1}/{R*K} selesai')

def metrik(yt, yp):
    return dict(RMSE=np.sqrt(mean_squared_error(yt, yp)), MAE=mean_absolute_error(yt, yp),
                R2=r2_score(yt, yp), MAPE=100 * mean_absolute_percentage_error(yt, yp))

baris = []
for n, P in oof.items():
    per_r = pd.DataFrame([metrik(y, P[r]) for r in range(R)])
    baris.append({'Model': n,
                  **{f'{k} (t/ha)': f'{per_r[k].mean():.3f} ± {per_r[k].std():.3f}' for k in ['RMSE', 'MAE']},
                  'R²': f"{per_r['R2'].mean():.3f} ± {per_r['R2'].std():.3f}",
                  'MAPE (%)': f"{per_r['MAPE'].mean():.2f} ± {per_r['MAPE'].std():.2f}",
                  'Waktu total (s)': round(waktu[n], 1)})
tabel1 = pd.DataFrame(baris)
tabel1.to_csv(HASIL / 'G_Tabel1_metrik_model.csv', index=False)
json.dump({k: [str(p) for p in v] for k, v in param_terpilih.items()}, open(HASIL / 'G_hyperparameter_terpilih.json', 'w'), indent=1)
tabel1

In [ ]:
# Prediksi vs aktual (rata-rata prediksi out-of-fold dari 10 pengulangan) dan distribusi galat
fig, ax = plt.subplots(2, len(oof), figsize=(4 * len(oof), 8))
lim = [0, max(y.max(), max(P.mean(0).max() for P in oof.values())) * 1.1]
for j, (n, P) in enumerate(oof.items()):
    yp = P.mean(0)
    ax[0, j].scatter(y, yp, alpha=.7); ax[0, j].plot(lim, lim, 'k--'); ax[0, j].set(xlim=lim, ylim=lim, title=n,
        xlabel='Aktual (t/ha)', ylabel='Prediksi (t/ha)')
    ax[1, j].hist(yp - y, bins=10); ax[1, j].set(xlabel='Galat (t/ha)', ylabel='Jumlah petak')
plt.tight_layout(); plt.savefig(HASIL / 'G_prediksi_vs_aktual.png', dpi=300); plt.show()

In [ ]:
# Learning curve CNN (untuk Reviewer E): konfigurasi yang paling sering terpilih, validasi 20%
g = pd.Series([json.dumps(p) for p in param_terpilih['CNN']]).mode()[0]; g = json.loads(g)
sc = StandardScaler().fit(X); Z = sc.transform(X)[..., None]
h = build_cnn(Z.shape[1], **g).fit(Z, y, validation_split=.2, epochs=300, batch_size=8, verbose=0)
plt.plot(h.history['loss'], label='Training loss'); plt.plot(h.history['val_loss'], label='Validation loss')
plt.xlabel('Epoch'); plt.ylabel('MSE (t/ha)²'); plt.title(f'Learning curve CNN {g}'); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(HASIL / 'G_learning_curve_CNN.png', dpi=300); plt.show()

## Ringkasan keluaran

Semua file tersimpan di `MyDrive/Data Padi/hasil/`:

- `B_metadata_citra.csv`, `B_foto_duplikat.csv`: parameter akuisisi untuk Metode (kamera, tinggi terbang, GSD, tanggal) dan daftar foto ganda yang dibuang
- `C_metrik_segmentasi_test.csv`: precision, recall, dan mAP segmentasi
- `E_sebaran_dataset_fase.csv`, `E_metrik_klasifikasi_test.csv`, `E_confusion_matrix.png`
- `E2_cv_per_fold.csv`, `E2_cv_metrik_per_kelas.csv`, `E2_cv_confusion_matrix.png`: hasil cross-validation 5-fold (angka utama untuk naskah)
- `F_fitur_petak.csv`, `F_overlay/`: luas, fase, dan indeks RGB per petak
- `G_Tabel1_metrik_model.csv`: **Tabel 1 untuk Reviewer B dan D** (RMSE, MAE, R², MAPE)
- `G_hyperparameter_terpilih.json`: untuk Reviewer C dan E
- `G_prediksi_vs_aktual.png`, `G_learning_curve_CNN.png`: gambar untuk Hasil

Angka di file-file ini menjadi sumber Tabel 2–4 dan Gambar di naskah.